In [2]:
# creating and visualizing tokens

In [3]:
import numpy as np

import requests
import re

In [4]:
book = requests.get('https://www.gutenberg.org/cache/epub/35/pg35.txt')
text = book.text

print(type(text))
print(len(text))

#print(text[:2000])  # print the first 1000 characters of the text

<class 'str'>
202504


In [5]:
'â\x80\x9d'.encode('latin1').decode('utf8') 

'”'

In [6]:
# character strings to replace with space
strings2replace = [
                '\r\n\r\nâ\x80\x9c', # new paragraph
                'â\x80\x9c',         # open quote
                'â\x80\x9d',         # close quote
                '\r\n',              # new line
                'â\x80\x94',         # hyphen
                'â\x80\x99',         # single apostrophe
                'â\x80\x98',         # single quote
                '_',                 # underscore, used for stressing
                ]

# e.g., 'â\x80\x9d'.encode('latin1').decode('utf8')

In [7]:
# use regular expression (re) to replace those strings with space
for str2match in strings2replace:
  regexp = re.compile(r'%s'%str2match)
  text = regexp.sub(' ',text)

# remove non-ASCII characters
text = re.sub(r'[^\x00-\x7F]+', ' ', text)

# remove numbers
text = re.sub('\d+','',text)

# and make everything lower-case
text = text.lower()

# let's have a look!
text[:2000]

<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_19297/3033343314.py:10: SyntaxWarning: invalid escape sequence '\d'
  text = re.sub('\d+','',text)


'the project gutenberg ebook of the time machine      this ebook is for the use of anyone anywhere in the united states and most other parts of the world at no cost and with almost no restrictions whatsoever. you may copy it, give it away or re-use it under the terms of the project gutenberg license included with this ebook or online at www.gutenberg.org. if you are not located in the united states, you will have to check the laws of the country where you are located before using this ebook.  title: the time machine  author: h. g. wells            release date: october ,  [ebook #]                 most recently updated: june ,   language: english  other information and formats: www.gutenberg.org/ebooks/   *** start of the project gutenberg ebook the time machine ***     the time machine  an invention  by h. g. wells   contents   i introduction  ii the machine  iii the time traveller returns  iv time travelling  v in the golden age  vi the sunset of mankind  vii a sudden shock  viii exp

In [8]:
# split by punctuation
import string
print(string.punctuation)
puncts4re = f'[{string.punctuation}\s]+'

words = re.split(puncts4re,text)
words = [item.strip() for item in words if item.strip()]

# remove single-character words
words = [item for item in words if len(item)>1]

# let's have a look!
words[:50]



!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_19297/1471106792.py:4: SyntaxWarning: invalid escape sequence '\s'
  puncts4re = f'[{string.punctuation}\s]+'


['the',
 'project',
 'gutenberg',
 'ebook',
 'of',
 'the',
 'time',
 'machine',
 'this',
 'ebook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'in',
 'the',
 'united',
 'states',
 'and',
 'most',
 'other',
 'parts',
 'of',
 'the',
 'world',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever',
 'you',
 'may',
 'copy',
 'it',
 'give',
 'it',
 'away',
 'or',
 're',
 'use',
 'it',
 'under',
 'the']

In [9]:
# create the vocab! (set of unique words)
vocab = sorted(set(words))

# convenience variables for later
nWords = len(words)
nLex = len(vocab)

print(f'{nWords} words')
print(f' {nLex} unique tokens')

33607 words
 4919 unique tokens


In [10]:
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for i,w in enumerate(vocab)}

# print out a few
for i in list(word2idx.items())[0:10000:87]:
  print(i)

('abandon', 0)
('age', 87)
('anticipated', 174)
('aspect', 261)
('bearing', 348)
('blood', 435)
('burrows', 522)
('century', 609)
('cleaned', 696)
('conceive', 783)
('corporation', 870)
('customary', 957)
('delivery', 1044)
('dining', 1131)
('dotted', 1218)
('ebbed', 1305)
('entry', 1392)
('explorings', 1479)
('fee', 1566)
('flickered', 1653)
('framed', 1740)
('get', 1827)
('griffins', 1914)
('having', 2001)
('hopes', 2088)
('import', 2175)
('inkling', 2262)
('isn', 2349)
('largely', 2436)
('lilies', 2523)
('malachite', 2610)
('mile', 2697)
('multiply', 2784)
('noises', 2871)
('operation', 2958)
('palps', 3045)
('persistently', 3132)
('pork', 3219)
('prohibition', 3306)
('ragged', 3393)
('refrain', 3480)
('rests', 3567)
('rush', 3654)
('seem', 3741)
('shock', 3828)
('slake', 3915)
('somewhere', 4002)
('stars', 4089)
('studs', 4176)
('sweeping', 4263)
('therefore', 4350)
('tore', 4437)
('twenty', 4524)
('urchin', 4611)
('volatile', 4698)
('wheel', 4785)
('world', 4872)


In [11]:
# encoder function (using for-loop instead of list-comp)
def encoder(words,encode_dict):

  # initialize a vector of numerical indices
  idxs = np.zeros(len(words),dtype=int)

  # loop through the words and find their token in the vocab
  for i,w in enumerate(words):
    idxs[i] = encode_dict[w]

  # return the indices!
  return idxs


# also need a decoder function
def decoder(idxs,decode_dict):
  return ' '.join([decode_dict[i] for i in idxs])

In [14]:
print(encoder(['the','time', 'machine'], word2idx))
print(decoder([0, 300, 2], idx2word))

[4339 4406 2594]
abandon avoidance abide
